# 5. CTCF Demo Pipeline

End-to-end demonstration of the ArChIPelago pipeline for a single transcription factor (CTCF_HUMAN):

1. Load FASTA sequences (positive ChIP-seq peaks + matched negative sequences)
2. Scan with SARUS PWMs (mono + di-nucleotide)
3. Build feature matrix, select top-1000 features
4. Train RandomForestClassifier
5. Evaluate with ROC-AUC and PR-AUC
6. Predict on held-out test sequences
7. Publication-quality figures

**Requirements:** Activate the ArChIPelago conda environment and edit `config.yml` before running.

```bash
conda activate ArChIPelago
```

**Data:** Download CTCF FASTA sequences and PWMs from Zenodo (DOI: 10.5281/zenodo.14927304).

## 0. Setup

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

# Add ArChIPelago_code to path so 'archipielago' package is importable
sys.path.insert(0, str(Path(os.getcwd())))

from archipielago.config import load_config, get_path
from archipielago import io, scanning, training

matplotlib.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'figure.dpi': 100,
})

# Demo TF
TF = 'CTCF_HUMAN'
TF_SHORT = TF.split('_')[0]   # 'CTCF'
print(f'Demo TF: {TF}')

In [ ]:
# Load configuration — edit config.yml to set your paths
cfg = load_config()  # reads config.yml next to this notebook

# Resolve paths
REPO_ROOT = Path(os.getcwd()).parent
RELEASE_DIR = Path(cfg['paths']['output_dir'])
SARUS_JAR  = Path(cfg['tools']['sarus_jar'])
JAVA_BIN   = cfg['tools']['java_bin']

# PWM directories (from HOCOMOCO v11)
PWM_MONO_DIR = REPO_ROOT / 'hocomoco11' / 'models' / 'pwm' / 'mono' / 'all' / f'{TF}'
PWM_DI_DIR   = REPO_ROOT / 'hocomoco11' / 'models' / 'pwm' / 'di'   / 'all' / f'{TF}'

# FASTA input files — expected to come from Zenodo or Notebook 0 output
FASTA_TRAIN_POS = RELEASE_DIR / 'train' / f'{TF}.PEAKS000001.macs.train.mfa'
FASTA_TEST_POS  = RELEASE_DIR / 'test'  / f'{TF}.PEAKS000001.macs.test.mfa'
# Negative (background) sequences — produced by BiasAway (Notebook 2) or from Zenodo
FASTA_TRAIN_NEG = RELEASE_DIR / 'train' / f'{TF}_negative_train.fasta'
FASTA_TEST_NEG  = RELEASE_DIR / 'test'  / f'{TF}_negative_test.fasta'

# Output directory for this demo
DEMO_DIR = RELEASE_DIR / 'demo' / TF
DEMO_DIR.mkdir(parents=True, exist_ok=True)
SCAN_DIR = DEMO_DIR / 'scans'
SCAN_DIR.mkdir(exist_ok=True)
(SCAN_DIR / 'mono').mkdir(exist_ok=True)
(SCAN_DIR / 'di').mkdir(exist_ok=True)

print(f'Release dir: {RELEASE_DIR}')
print(f'SARUS jar: {SARUS_JAR} (exists: {SARUS_JAR.exists()})')
print(f'PWM mono: {PWM_MONO_DIR} (exists: {PWM_MONO_DIR.exists()})')
print(f'Train FASTA (pos): {FASTA_TRAIN_POS} (exists: {FASTA_TRAIN_POS.exists()})')

## 1. Load Input Sequences

In [ ]:
# Load positive (ChIP-seq peaks) and negative (background) sequences
train_pos = io.load_fasta(FASTA_TRAIN_POS)
train_neg = io.load_fasta(FASTA_TRAIN_NEG)
test_pos  = io.load_fasta(FASTA_TEST_POS)
test_neg  = io.load_fasta(FASTA_TEST_NEG)

print(f'Train: {len(train_pos)} positive, {len(train_neg)} negative')
print(f'Test:  {len(test_pos)} positive, {len(test_neg)} negative')

# Balance: use equal numbers of pos/neg
SIZE = min(len(train_pos), len(train_neg))
rng = np.random.default_rng(42)
train_pos_sel = rng.choice(len(train_pos), SIZE, replace=False).tolist()
train_neg_sel = rng.choice(len(train_neg), SIZE, replace=False).tolist()

train_records = [train_pos[i] for i in train_pos_sel] + [train_neg[i] for i in train_neg_sel]
train_labels  = [1] * SIZE + [0] * SIZE

test_records  = test_pos + test_neg
test_labels   = [1] * len(test_pos) + [0] * len(test_neg)

print(f'Balanced train set: {len(train_records)} sequences ({SIZE} pos + {SIZE} neg)')
print(f'Test set: {len(test_records)} sequences')

In [ ]:
# GC content distribution
from Bio.SeqUtils import gc_fraction

gc_pos = [gc_fraction(seq) * 100 for _, seq in train_pos[:SIZE]]
gc_neg = [gc_fraction(seq) * 100 for _, seq in train_neg[:SIZE]]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(gc_pos, bins=40, alpha=0.6, label='Positive (ChIP-seq peaks)', color='steelblue')
ax.hist(gc_neg, bins=40, alpha=0.6, label='Negative (background)', color='salmon')
ax.set_xlabel('GC content (%)')
ax.set_ylabel('Count')
ax.set_title(f'{TF_SHORT} — Sequence GC content')
ax.legend()
plt.tight_layout()
plt.savefig(DEMO_DIR / f'{TF_SHORT}_GC_distribution.pdf', dpi=150)
plt.show()

## 2. PWM Scanning with SARUS

In [ ]:
# Write combined FASTA files for scanning
TRAIN_FASTA = DEMO_DIR / 'train_combined.fasta'
TEST_FASTA  = DEMO_DIR / 'test_combined.fasta'
io.save_fasta(train_records, TRAIN_FASTA)
io.save_fasta(test_records, TEST_FASTA)
print(f'Written: {TRAIN_FASTA}')
print(f'Written: {TEST_FASTA}')

In [ ]:
# Scan with mononucleotide PWMs
mono_pwm_files = sorted(PWM_MONO_DIR.glob('*.pwm'))
print(f'Found {len(mono_pwm_files)} mono PWMs for {TF_SHORT}')

for pwm_file in mono_pwm_files:
    out_file = SCAN_DIR / 'mono' / f'{pwm_file.stem}.txt'
    if out_file.exists():
        continue  # skip if already scanned
    print(f'  Scanning {pwm_file.name}...', end=' ')
    scanning.run_sarus(
        fasta_path=TRAIN_FASTA,
        pwm_path=pwm_file,
        sarus_jar=SARUS_JAR,
        output_path=out_file,
        java_bin=JAVA_BIN,
        pwm_type='mono',
    )
    print('done')

print('Mono scanning complete.')

In [ ]:
# Scan with dinucleotide PWMs
di_pwm_files = sorted(PWM_DI_DIR.glob('*.dpwm'))
print(f'Found {len(di_pwm_files)} di PWMs for {TF_SHORT}')

for pwm_file in di_pwm_files:
    out_file = SCAN_DIR / 'di' / f'{pwm_file.stem}.txt'
    if out_file.exists():
        continue
    print(f'  Scanning {pwm_file.name}...', end=' ')
    scanning.run_sarus(
        fasta_path=TRAIN_FASTA,
        pwm_path=pwm_file,
        sarus_jar=SARUS_JAR,
        output_path=out_file,
        java_bin=JAVA_BIN,
        pwm_type='di',
    )
    print('done')

print('Di scanning complete.')

## 3. Build Feature Matrix and Select Top Features

In [ ]:
# Build combined mono+di feature matrix
X_all = scanning.build_feature_matrix(SCAN_DIR, mode='mono_di')
y_all = np.array(train_labels)

print(f'Feature matrix shape: {X_all.shape}')
print(f'  Positive sequences: {y_all.sum()}')
print(f'  Negative sequences: {(y_all == 0).sum()}')

In [ ]:
# Select top 1000 features by Random Forest importance
N_FEATURES = 1000
top_features = scanning.select_top_features(X_all, y_all, n=N_FEATURES)
X_train = X_all[top_features]
print(f'Selected {len(top_features)} features')
print(f'Top 10 features: {top_features[:10]}')

## 4. Train RandomForestClassifier

In [ ]:
from archipielago.training import train_rf, evaluate_model, cross_validate_model

# Train model (adjust n_jobs to your machine; n_jobs=40 for full run)
model = train_rf(
    X_train, y_all,
    n_estimators=100,
    max_depth=6,
    n_jobs=4,
    random_state=42,
)
print('Model trained.')
print(f'  n_estimators: {model.n_estimators}')
print(f'  n_features_in_: {model.n_features_in_}')

In [ ]:
# Feature importance plot (top 20)
importances = pd.Series(
    model.feature_importances_, index=top_features
).nlargest(20)

fig, ax = plt.subplots(figsize=(8, 5))
importances[::-1].plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Feature importance')
ax.set_title(f'{TF_SHORT} — Top 20 feature importances')
plt.tight_layout()
plt.savefig(DEMO_DIR / f'{TF_SHORT}_feature_importances.pdf', dpi=150)
plt.show()

## 5. Evaluation on Test Set

In [ ]:
# Build test feature matrix
# First scan test sequences
SCAN_DIR_TEST = DEMO_DIR / 'scans_test'
(SCAN_DIR_TEST / 'mono').mkdir(parents=True, exist_ok=True)
(SCAN_DIR_TEST / 'di').mkdir(exist_ok=True)

for pwm_file in mono_pwm_files:
    out_file = SCAN_DIR_TEST / 'mono' / f'{pwm_file.stem}.txt'
    if not out_file.exists():
        scanning.run_sarus(TEST_FASTA, pwm_file, SARUS_JAR, out_file,
                           java_bin=JAVA_BIN, pwm_type='mono')

for pwm_file in di_pwm_files:
    out_file = SCAN_DIR_TEST / 'di' / f'{pwm_file.stem}.txt'
    if not out_file.exists():
        scanning.run_sarus(TEST_FASTA, pwm_file, SARUS_JAR, out_file,
                           java_bin=JAVA_BIN, pwm_type='di')

X_test_all = scanning.build_feature_matrix(SCAN_DIR_TEST, mode='mono_di')
X_test = X_test_all[top_features]
y_test = np.array(test_labels)
print(f'Test feature matrix: {X_test.shape}')

In [ ]:
# Evaluate
results = evaluate_model(model, X_test, y_test)
print(f"Test ROC-AUC : {results['roc_auc']:.4f}")
print(f"Test PR-AUC  : {results['pr_auc']:.4f}")

## 6. Predictions on Held-Out Sequences

In [ ]:
# Predict probability on each test sequence
y_pred_proba = model.predict_proba(X_test)[:, 1]

pred_df = pd.DataFrame({
    'sequence_id': [h for h, _ in test_records],
    'label': y_test,
    'predicted_probability': y_pred_proba,
}).sort_values('predicted_probability', ascending=False).reset_index(drop=True)

pred_df.to_csv(DEMO_DIR / f'{TF_SHORT}_predictions.tsv', sep='\t', index=False)
print('Top 10 predictions:')
pred_df.head(10)

## 7. Publication-Quality Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# ROC curve
ax = axes[0]
ax.plot(results['fpr'], results['tpr'],
        color='steelblue', lw=2,
        label=f"RF mono+di (AUC = {results['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'{TF_SHORT} ROC Curve')
ax.legend(loc='lower right', fontsize=10)

# PR curve
ax = axes[1]
ax.plot(results['recall'], results['precision'],
        color='tomato', lw=2,
        label=f"RF mono+di (AUC = {results['pr_auc']:.3f})")
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'{TF_SHORT} Precision-Recall Curve')
ax.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.savefig(DEMO_DIR / f'{TF_SHORT}_ROC_PR.pdf', dpi=150)
plt.show()
print(f'Figure saved to {DEMO_DIR / (TF_SHORT + "_ROC_PR.pdf")}')

In [ ]:
# Score distribution: positives vs negatives
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(y_pred_proba[y_test == 1], bins=50, alpha=0.7,
        label='Positive (ChIP-seq peaks)', color='steelblue', density=True)
ax.hist(y_pred_proba[y_test == 0], bins=50, alpha=0.7,
        label='Negative (background)', color='salmon', density=True)
ax.set_xlabel('Predicted binding probability')
ax.set_ylabel('Density')
ax.set_title(f'{TF_SHORT} — Score distribution')
ax.legend()
plt.tight_layout()
plt.savefig(DEMO_DIR / f'{TF_SHORT}_score_distribution.pdf', dpi=150)
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| TF | CTCF_HUMAN |
| Train size (pos+neg) | {SIZE}+{SIZE} |
| Features | 1000 mono+di PWM scores |
| Model | RandomForestClassifier |
| Test ROC-AUC | see above |
| Test PR-AUC | see above |

To run on a different TF, change `TF = 'CTCF_HUMAN'` in the Setup cell.
All 36 supported TFs are listed in `ArChIPelago-TFBS-finder/scanning_tool.py`.